# 模块 9 — PyVista 交互式三维体渲染演示

本 notebook 演示如何在 **Jupyter 中交互式**渲染 `GoldakFDM` 的三维焊接温度场
(熔合区 / HAZ 等温面 + 对称面温度切片)。可直接用鼠标拖拽旋转、缩放、平移。

## 前置条件

安装交互式 notebook 所需依赖 (含 pyvista + jupyter):

```bash
uv sync --extra notebook
uv run jupyter lab      # 或 uv run jupyter notebook
```

> `render(..., notebook=True)` 会返回 PyVista `Plotter` 对象供内联显示，
> 而不会弹出独立 GUI 窗口。

## 0. 选择内联后端

- `'trame'` — 浏览器内完整 VTK 交互，推荐用于 JupyterLab。
- `'static'` — 静态图，无交互，适合作为无显示环境的回退。

In [ ]:
import pyvista as pv

pv.set_jupyter_backend('trame')

## 1. 求解三维瞬态温度场

用默认工艺参数运行 `GoldakFDM` (半对称模型)。`run()` 同时保存末时刻温度场
`g.T` 与逐步累积的峰值温度场 `g.peak` (用于划分熔合区 / HAZ)。

In [ ]:
from welding_dynamics import GoldakFDM

g = GoldakFDM()
g.run(t_end=5.0)

L, W, D = g.pool_size()
print(f'网格 {g.Nx}x{g.Ny}x{g.Nz} (半模型)')
print(f'熔池 L×W×D = {L:.1f}×{W:.1f}×{D:.1f} mm')
print(f'峰值温度 T_max = {g.peak.max():.0f} K')

## 2. 交互渲染峰值温度场 (熔合区 + HAZ)

`render(g, field='peak', notebook=True)` 将半模型沿 y=0 镜像为全熔池，绘制
熔点等温面 (红，熔合线) 与 HAZ 等温面 (橙，半透明) 及对称面温度切片，
并返回 Plotter 内联显示。**用鼠标拖拽旋转视角。**

In [ ]:
from welding_dynamics import render

fig = render(g, field='peak', notebook=True)
fig

## 3. 交互渲染末时刻温度场

同一函数换成 `field='final'` 渲染热源扫过后某一时刻的温度分布。

In [ ]:
fig2 = render(g, field='final', notebook=True)
fig2

## 4. 自定义场景 (底层 PyVista API)

若想交互式调节等温面阈值 / 透明度 / 增加体渲染，可直接用 PyVista
搭建场景。下面把半模型镜像为全场后，叠加可调的多条等温面。

In [ ]:
import numpy as np
import pyvista as pv

# 半模型 (y>=0) 沿 y=0 镜像为全场; z 取负 => 向下为深度
T = g.peak
Tfull = np.concatenate([T[:, :0:-1, :], T], axis=1)
x = g.x * 1e3
y = np.concatenate([-g.y[:0:-1], g.y]) * 1e3
z = -g.z * 1e3
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

grid = pv.StructuredGrid(X, Y, Z)
grid.point_data['T'] = Tfull.ravel(order='F')
plotter = pv.Plotter(window_size=(900, 600))
plotter.set_background('white')

# 多条等温面: 熔点 / 1500 / 1073 K(HAZ)
plotter.add_mesh(grid.contour([float(g.Tm)], scalars='T'), color=(0.85, 0.1, 0.1), opacity=0.6)
plotter.add_mesh(grid.contour([1500.0], scalars='T'), color=(1.0, 0.5, 0.0), opacity=0.3)
plotter.add_mesh(grid.contour([1073.0], scalars='T'), color=(1.0, 0.85, 0.2), opacity=0.2)

# 顶面切片, 便于读熔深
top_slice = grid.slice(normal='z', origin=(0.0, 0.0, 0.0))
plotter.add_mesh(top_slice, scalars='T', cmap='jet', opacity=0.82,
                 scalar_bar_args={'title': 'T [K]'})
plotter.add_axes(xlabel='x [mm]', ylabel='y [mm]', zlabel='depth [mm]')
plotter.show_bounds(xtitle='x [mm]', ytitle='y [mm]', ztitle='depth [mm]')
plotter.view_isometric()
plotter

## 相关

- **脚本/无头环境**: 用 `render(g, offscreen=True, outfile='results/m9.png')` 离屏存图;
  或命令行 `uv run welding-sim-3d` (求解 → 导出 OpenFOAM → 离屏截图)。
- **OpenFOAM 导出**: `from welding_dynamics import export_openfoam; export_openfoam(g)`
  写出可在 ParaView 直接打开的算例 (`results/openfoam_case/case.foam`)。